In [1]:
import subprocess, sys, shutil, json, hashlib, time
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics'])
import torch, yaml, numpy as np, pandas as pd
from PIL import Image
from ultralytics import YOLO
assert torch.cuda.is_available(), 'فعّلي GPU'

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/out'); OUT.mkdir(exist_ok=True, parents=True)
CLASSES = ['D00','D10','D20','D40']

# حماية: لا تدخل أي بيانات محلية
bad = [p.name for p in INPUT.glob('*/*') if 'local' in p.name.lower() or 'my-photos' in p.name.lower()]
assert not bad, f'مدخلات محلية مرفقة، احذفيها: {bad}'

# --- إيجاد البيانات تلقائيًا ---
SRC = next(p for p in INPUT.rglob('sample9000') if (p/'train/images').is_dir() and (p/'val/images').is_dir())
print('SRC:', SRC)
ROOT = Path('/tmp/data')
for s in ['train','val']:
    shutil.copytree(SRC/s, ROOT/s, dirs_exist_ok=True)
DATA = ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT), train='train/images', val='val/images', names=CLASSES)))
print('train', len(list((ROOT/'train/images').glob('*.jpg'))), 'val', len(list((ROOT/'val/images').glob('*.jpg'))))

# --- إيجاد الأوزان تلقائيًا ---
pts = list(INPUT.rglob('*.pt'))
print('كل ملفات .pt الموجودة:'); [print('  ', p) for p in pts]
W = {}
for p in pts:
    if p.name == 'run17_best_finetuned.pt': W['17'] = p
    if p.name == 'best.pt' and '14-roadscan-team-v2' in str(p): W['14'] = p
print('الأوزان المعتمدة:', W)
assert W, 'ما لقيت أوزان، تأكدي من إضافة مخرجات دفتري 14 و17'

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
SRC: /kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2/sample9000
train 8999 val 1977
كل ملفات .pt الموجودة:
   /kaggle/input/notebooks/rimasabunasser/14-roadscan-team-v2-training/yolo11s.pt
   /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/artifacts_run17/best.pt
   /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/artifacts_run17/last.pt
   /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/artifacts_run17/published_yolov8s_rdd2022_seed0.pt
   /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17_model_comparison/original_published.pt
   /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17

In [2]:
rows = []
for name, w in W.items():
    net = YOLO(str(w))
    assert [net.names[i] for i in range(4)] == CLASSES, (name, net.names)
    for sz in [640, 960]:
        m = net.val(data=str(DATA), split='val', imgsz=sz, batch=8, conf=0.001, iou=0.7,
                    max_det=300, plots=False, verbose=False)
        rows.append(dict(model=name, imgsz=sz, mAP50=m.box.map50, mAP50_95=m.box.map, P=m.box.mp, R=m.box.mr))
        print(rows[-1], flush=True)
    del net; torch.cuda.empty_cache()
base = pd.DataFrame(rows); base.to_csv(OUT/'baseline.csv', index=False); display(base)

b960 = base[base.imgsz == 960].sort_values('mAP50_95', ascending=False)
START_NAME = b960.iloc[0].model
START = W[START_NAME]
print('نقطة البداية:', START_NAME, START, flush=True)

Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1442.3±623.0 MB/s, size: 79.0 KB)
val: Scanning /tmp/data/val/labels... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 1.1Kit/s 1.7s
val: New cache created: /tmp/data/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 248/248 10.1it/s 24.7s
                   all       1977       4650      0.917      0.926      0.959      0.658
Speed: 0.8ms preprocess, 8.9ms inference, 0.0ms loss, 0.8ms postprocess per image
{'model': '17', 'imgsz': 640, 'mAP50': np.float64(0.9593002182315967), 'mAP50_95': np.float64(0.6578987935389327), 'P': np.float64(0.9173666742017224), 'R': np.float64(0.9255760963761883)}
Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Mode

,model,imgsz,mAP50,mAP50_95,P,R
0,17,640,0.959300,0.657899,0.917367,0.925576
1,17,960,0.869372,0.469599,0.810501,0.840997
2,14,640,0.227095,0.087325,0.297812,0.304142
3,14,960,0.140900,0.040336,0.230590,0.232276


نقطة البداية: 17 /kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17_model_comparison/run17_best_finetuned.pt


In [3]:
TRAIN_HOURS = 3.0   # عدّليه حسب وقتك (سقف التدريب بالساعات)
net = YOLO(str(START))
net.train(
    data=str(DATA), imgsz=960, epochs=30, time=TRAIN_HOURS, patience=8,
    batch=8, freeze=10, optimizer='AdamW', lr0=0.0002, lrf=0.1, cos_lr=True,
    warmup_epochs=1, mosaic=0.5, close_mosaic=5, scale=0.4,
    seed=42, device=0, workers=2, amp=True, save_period=-1,
    project='/kaggle/working/training', name='run24_960', exist_ok=True)
NEW = Path('/kaggle/working/training/run24_960/weights/best.pt')
assert NEW.is_file(), 'ما انحفظ best.pt'
del net; torch.cuda.empty_cache()

Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17_model_comparison/run17_best_finetuned.pt, momentum=0.937, mos

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/61      2.72G      1.658      1.502      1.697         20        960: 100% ━━━━━━━━━━━━ 1125/1125 7.5it/s 2:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 5.1it/s 24.1s
                   all       1977       4650      0.826      0.847      0.883      0.495

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/62      2.72G      1.637      1.479       1.69         15        960: 100% ━━━━━━━━━━━━ 1125/1125 7.5it/s 2:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 5.1it/s 24.1s
                   all       1977       4650      0.817      0.837      0.876      0.488

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/62      2.72G      1.639      1.472      1.692         12        960: 100% ━━━━━━━━━━━━ 1125/1125 7.5it/s 2:30
                 Class     Images  Instances

In [4]:
cands = dict(W); cands['new_run24'] = NEW
rows = []
for name, w in cands.items():
    net = YOLO(str(w))
    for sz in [640, 960]:
        m = net.val(data=str(DATA), split='val', imgsz=sz, batch=8, conf=0.001, iou=0.7,
                    max_det=300, plots=False, verbose=False)
        rows.append(dict(model=name, imgsz=sz, mAP50=m.box.map50, mAP50_95=m.box.map, P=m.box.mp, R=m.box.mr))
        print(rows[-1], flush=True)
    del net; torch.cuda.empty_cache()
final = pd.DataFrame(rows); final.to_csv(OUT/'final_comparison.csv', index=False); display(final)

best = final.sort_values('mAP50_95', ascending=False).iloc[0]
winner = cands[best.model]
shutil.copy2(winner, OUT/'final_weights.pt')
cfg = dict(model=best.model, source_weights=str(winner), imgsz=int(best.imgsz), conf=0.46,
           nms_iou=0.7, max_det=300, matching_iou=0.5,
           sha256=hashlib.sha256((OUT/'final_weights.pt').read_bytes()).hexdigest(),
           val_mAP50=float(best.mAP50), val_mAP50_95=float(best.mAP50_95),
           selection='China validation only (mAP50-95)', local_images_used=False)
(OUT/'final_config.json').write_text(json.dumps(cfg, indent=2))
print('الفائز:', cfg, flush=True)

# --- Precision/Recall/F1 عند ثقة 0.46 على China val (اختياري، لا يوقف التشغيل لو فشل) ---
try:
    def iou1(a, bs):
        lo = np.maximum(a[:2], bs[:, :2]); hi = np.minimum(a[2:], bs[:, 2:])
        inter = np.maximum(hi - lo, 0).prod(1)
        return inter / np.maximum((a[2:]-a[:2]).prod() + (bs[:, 2:]-bs[:, :2]).prod(1) - inter, 1e-9)
    net = YOLO(str(OUT/'final_weights.pt')); tp = fp = fn = 0
    for ip in sorted((ROOT/'val/images').glob('*.jpg')):
        w, h = Image.open(ip).size
        g = []
        lp = ROOT/'val/labels'/(ip.stem + '.txt')
        for line in lp.read_text().splitlines():
            if line.strip():
                c, x, y, bw, bh = map(float, line.split())
                g.append([c, (x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h])
        g = np.array(g).reshape(-1, 5); used = set()
        r = net.predict(str(ip), imgsz=cfg['imgsz'], conf=0.46, iou=0.7, max_det=300, verbose=False)[0]
        d = r.boxes.data.cpu().numpy(); d = d[np.argsort(-d[:, 4])] if len(d) else d
        for pp in d:
            ids = [i for i in range(len(g)) if i not in used and int(g[i, 0]) == int(pp[5])]
            hit = False
            if ids:
                ov = iou1(pp[:4], g[ids, 1:]); k = int(ov.argmax())
                if ov[k] >= 0.5: used.add(ids[k]); hit = True
            tp += hit; fp += (not hit)
        fn += len(g) - len(used)
    P = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1); F1 = 2*tp / max(2*tp+fp+fn, 1)
    print(f'China val @0.46: P={P:.3f} R={R:.3f} F1={F1:.3f}')
    (OUT/'val_at_046.json').write_text(json.dumps(dict(P=P, R=R, F1=F1, TP=tp, FP=fp, FN=fn)))
except Exception as e:
    print('تخطيت حساب 0.46:', e)
print('DONE', flush=True)

Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1853.2±276.4 MB/s, size: 79.0 KB)
val: Scanning /tmp/data/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 691.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 248/248 9.4it/s 26.4s
                   all       1977       4650      0.917      0.926      0.959      0.658
Speed: 0.9ms preprocess, 9.9ms inference, 0.0ms loss, 0.8ms postprocess per image
{'model': '17', 'imgsz': 640, 'mAP50': np.float64(0.9593002182315967), 'mAP50_95': np.float64(0.6578987935389327), 'P': np.float64(0.9173666742017224), 'R': np.float64(0.9255760963761883)}
Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,132 par

,model,imgsz,mAP50,mAP50_95,P,R
0,17,640,0.959300,0.657899,0.917367,0.925576
1,17,960,0.869372,0.469599,0.810501,0.840997
2,14,640,0.227095,0.087325,0.297812,0.304142
3,14,960,0.140900,0.040336,0.230590,0.232276
4,new_run24,640,0.954846,0.644497,0.903976,0.920657
5,new_run24,960,0.882347,0.495591,0.824847,0.847737


الفائز: {'model': '17', 'source_weights': '/kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17_model_comparison/run17_best_finetuned.pt', 'imgsz': 640, 'conf': 0.46, 'nms_iou': 0.7, 'max_det': 300, 'matching_iou': 0.5, 'sha256': '25b412ff18c05e2607f4bc61b872f39826e8d53ffdc7647654c1f419e355d15f', 'val_mAP50': 0.9593002182315967, 'val_mAP50_95': 0.6578987935389327, 'selection': 'China validation only (mAP50-95)', 'local_images_used': False}
China val @0.46: P=0.893 R=0.845 F1=0.868
DONE
